In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: A Qiskit Function by Global Data Quantum


> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ให้บริการเฉพาะสำหรับผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (via IBM Quantum Platform API) Plan เท่านั้น ขณะนี้อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง

# ภาพรวม
Quantum Portfolio Optimizer คือ Qiskit Function ที่ออกแบบมาเพื่อแก้ปัญหา dynamic portfolio optimization ซึ่งเป็นปัญหาพื้นฐานในด้านการเงินที่มุ่งปรับสัดส่วนการลงทุนเป็นระยะ ๆ ในสินทรัพย์หลายประเภท เพื่อเพิ่มผลตอบแทนและลดความเสี่ยง ด้วยการใช้เทคนิค quantum optimization ล้ำสมัย ฟังก์ชันนี้ทำให้กระบวนการง่ายขึ้น ผู้ใช้ที่ไม่มีความเชี่ยวชาญด้าน quantum computing ก็สามารถได้รับประโยชน์จากมันในการหา investment trajectory ที่ดีที่สุด เหมาะสำหรับผู้จัดการพอร์ตโฟลิโอ นักวิจัยด้าน quantitative finance และนักลงทุนทั่วไป เครื่องมือนี้รองรับการ back-testing กลยุทธ์การซื้อขายในการ optimize พอร์ตโฟลิโอ

# คำอธิบายฟังก์ชัน
Quantum Portfolio Optimizer ใช้อัลกอริทึม Variational Quantum Eigensolver (VQE) เพื่อแก้ปัญหา Quadratic Unconstrained Binary Optimization (QUBO) สำหรับ dynamic portfolio optimization ผู้ใช้เพียงแค่ระบุข้อมูลราคาสินทรัพย์และกำหนดเงื่อนไขการลงทุน จากนั้นฟังก์ชันจะรันกระบวนการ quantum optimization และคืนค่าชุด investment trajectory ที่ optimize แล้ว

กระบวนการประกอบด้วยสี่ขั้นตอนหลัก ขั้นแรก ข้อมูล input จะถูก map ไปยังปัญหาที่ compatible กับ quantum โดยสร้าง QUBO ของ dynamic portfolio optimization และแปลงเป็น quantum operator (Ising Hamiltonian) ต่อมา ปัญหา input และอัลกอริทึม VQE จะถูกปรับแต่งเพื่อรันบน quantum hardware จากนั้นรัน VQE บน quantum hardware และสุดท้ายผลลัพธ์จะถูก post-process เพื่อให้ได้ investment trajectory ที่ดีที่สุด ระบบยังมี noise-aware ([SQD](/guides/qiskit-addons-sqd)-based) post-processing เพื่อเพิ่มคุณภาพของ output ด้วย

Qiskit Function นี้อ้างอิงจาก [published manuscript](https://arxiv.org/abs/2412.19150) ของ Global Data Quantum

![Visualization of the workflow of the function](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)

# Input
อาร์กิวเมนต์ input ของฟังก์ชันมีรายละเอียดในตารางต่อไปนี้ ต้องระบุข้อมูลสินทรัพย์และรายละเอียดของปัญหา นอกจากนี้ยังสามารถกำหนด VQE settings เพื่อปรับแต่งกระบวนการ optimize ได้

| **ชื่อ**               | **ประเภท**           | **คำอธิบาย**                                          | **จำเป็น** | **ค่าเริ่มต้น**                          | **ตัวอย่าง**                              |
|------------------------|--------------------|----------------------------------------------------------|--------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Dictionary ที่เก็บราคาสินทรัพย์                         | ใช่          | -                                    | -                                        |
| qubo_settings          | json               | การตั้งค่าของ QUBO                                     | ใช่          | -                                    | ดูตัวอย่างในตารางด้านล่าง     |
| ansatz_settings        | json               | การตั้งค่าของ ansatz                                   | ไม่           | `None`                                | ดูตัวอย่างในตารางด้านล่าง     |
| optimizer_settings     | json               | การตั้งค่าของ optimizer                                | ไม่           | `None`                                | ดูตัวอย่างในตารางด้านล่าง     |
| backend                | str                | ชื่อ QPU Backend                                     | ไม่           |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | รายการ Session ID สำหรับดึงข้อมูลจากการรันก่อนหน้า[(*)](#1-note) | ไม่           | Empty list                           | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | ใช้ noise-aware SQD post-processing                      | ไม่           | `True`                                | `True`                                   |
| tags                   | list of strings    | รายการ tag สำหรับระบุ experiment                  | ไม่           | Empty list                           | `["optimization", "quantum_computing"]`  |

<span id="1-note"></span>
*หากต้องการรันต่อจากเดิมหรือดึง job ที่ประมวลผลใน Session ก่อนหน้า ให้ส่งรายการ Session ID ผ่านพารามิเตอร์ `previous_session_id` ซึ่งมีประโยชน์มากในกรณีที่ optimization task ล้มเหลวกลางคัน และต้องรันต่อจนเสร็จ ในการทำเช่นนั้น ต้องระบุอาร์กิวเมนต์เดิมที่ใช้ในการรันครั้งแรก พร้อมกับรายการ `previous_session_id` ตามที่อธิบาย*

> **Caution:** การโหลดข้อมูลจาก Session ก่อนหน้า (เพื่อรันต่อ) อาจใช้เวลาสูงสุดหนึ่งชั่วโมง

## `assets`
ข้อมูลต้องมีโครงสร้างเป็น JSON object ที่เก็บข้อมูลราคาปิดของสินทรัพย์ทางการเงินในวันที่ระบุ รูปแบบมีดังนี้

- Primary key (string): ชื่อหรือสัญลักษณ์ ticker ของสินทรัพย์ทางการเงิน (เช่น "8801.T")
- Secondary key (string): วันที่ในรูปแบบ YYYY-MM-DD
- Value (number): ราคาปิดของสินทรัพย์ในวันที่ระบุ สามารถป้อนราคาแบบ normalized หรือไม่ normalized ก็ได้

*โปรดทราบว่า dictionary ทั้งหมดต้องมี secondary key (วันที่) เหมือนกัน หากสินทรัพย์บางตัวไม่มีวันที่ที่สินทรัพย์อื่นมี ต้องเติมข้อมูลให้ครบเพื่อความสม่ำเสมอ เช่น สามารถใช้ราคาปิดล่าสุดที่บันทึกไว้ของสินทรัพย์นั้น*

### ตัวอย่าง
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** ข้อมูลสินทรัพย์ต้องมีราคาปิดอย่างน้อย ``(nt+1) * dt`` timestamps (เช่น วัน) (ดูส่วน [`qubo_settings`](#qubo_settings))

## `qubo_settings`
ตารางต่อไปนี้อธิบาย key ของ dictionary `qubo_settings` สร้าง dictionary โดยระบุจำนวน time steps `nt`, จำนวน resolution Qubit `nq`, และ `max_investment` หรือจะเปลี่ยนค่า default อื่น ๆ ก็ได้

| ชื่อ                | ประเภท    | คำอธิบาย                                                                 | จำเป็น | ค่าเริ่มต้น | ตัวอย่าง |
|---------------------|---------|-----------------------------------------------------------------------------|----------|---------|---------|
| nt                  | int     | จำนวน time steps                                                         | ใช่      | -       | 4       |
| nq                  | int     | จำนวน resolution Qubit                                                                | ใช่      | -       | 4       |
| max_investment      | float   | จำนวนสูงสุดของหน่วยเงินที่ลงทุนรวมทุกสินทรัพย์                           | ใช่      | -       | 10      |
| dt*                  | int     | ช่วงเวลาที่พิจารณาในแต่ละ time step หน่วยตรงกับช่วงเวลาระหว่าง key ในข้อมูลสินทรัพย์                                                 | ไม่       | 30      | -       |
| risk_aversion       | float     | ค่าสัมประสิทธิ์ความเกลียดความเสี่ยง                                   | ไม่       | 1000    | -       |
| transaction_fee     | float     | ค่าสัมประสิทธิ์ค่าธรรมเนียมการซื้อขาย                                                 | ไม่       | 0.01   | -       |
| restriction_coeff   | float   | Lagrange multiplier ที่ใช้บังคับเงื่อนไขของปัญหาใน QUBO formulation | ไม่       | 1       | -       |

## `ansatz_settings`
หากต้องการแก้ไข default options ให้สร้าง dictionary สำหรับพารามิเตอร์ `ansatz_settings` โดยมี key ดังต่อไปนี้ ค่าเริ่มต้นของ ansatz คือ `"real_amplitudes"` และทั้ง extra options (ดูตารางต่อไปนี้) ถูกตั้งเป็น `False`

| ชื่อ                  | ประเภท     | คำอธิบาย                                                                 | จำเป็น | ค่าเริ่มต้น |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|---------|
| ansatz[*](#single-star)                | str      | Ansatz ที่จะใช้                                             | ไม่      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | เปิดใช้งาน multiple passmanager subroutine (ไม่รองรับสำหรับ Tailored ansatz) | ไม่       | `False`   |
| dd_enable   | bool     | เพิ่ม dynamical decoupling                                    | ไม่       | `False`   |

<span id="single-star"></span>
\* Ansatz ที่รองรับ
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (เฉพาะ `ibm_torino` Backend, 7 สินทรัพย์, 4 time steps และ 4 resolution Qubit)

<span id="double-star"></span>
** ถ้า ``multiple_passmanager`` ถูกตั้งเป็น ``False`` ฟังก์ชันจะใช้ default Qiskit pass manager ที่มี `optimization_level=3` ถ้าตั้งเป็น ``True`` subroutine ``multiple_passmanager`` จะเปรียบเทียบ pass manager สามตัว ได้แก่ default Qiskit pass manager เดิม, pass manager ที่ map Qubit ไปยัง QPU first neighbors chain และ [AI transpiler](/guides/ai-transpiler-passes) services จากนั้นจะเลือก pass manager ที่มีค่า cumulative error ต่ำที่สุดโดยประมาณ

## `optimizer_settings`
พารามิเตอร์นี้คือ dictionary ที่มี options ที่ปรับแต่งได้สำหรับกระบวนการ optimize

| ชื่อ               | ประเภท   | คำอธิบาย                                     | จำเป็น | ค่าเริ่มต้น               |
|--------------------|--------|-------------------------------------------------|----------|-----------------------|
| primitive_options  | json   | การตั้งค่าของ primitive               | ไม่      | -                     |
| optimizer          | str    | classical optimizer ที่เลือก                            | ไม่       | `"differential_evolution"` |
| optimizer_options  | json   | การกำหนดค่าของ optimizer                  | ไม่       | -                     |

> **Note:** ปัจจุบัน optimizer option ที่รองรับมีเพียง `"differential_evolution"` เท่านั้น

ภายใต้ key `primitive_options` และ `optimizer_options` เราตั้งค่า dictionary ที่มีพารามิเตอร์ดังต่อไปนี้

### `primitive_options`
| ชื่อ              | ประเภท   | คำอธิบาย                                | จำเป็น | ค่าเริ่มต้น                    | ตัวอย่าง                    |
|-------------------|--------|--------------------------------------------|----------|----------------------------|----------------------------|
| sampler_shots     | int    | จำนวน shots ของ Sampler            | ไม่       | 100000                     | -                          |
| estimator_shots   | int    | จำนวน shots ของ Estimator          | ไม่       | 25000                      | -                          |
| estimator_precision | float | ความแม่นยำที่ต้องการของค่าคาดหวัง ถ้าระบุ จะใช้ precision แทน `estimator_shots` | ไม่ | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int or str | เวลาสูงสุดที่ runtime session จะเปิดอยู่ได้ก่อนถูกปิดโดยบังคับ สามารถกำหนดเป็นวินาที (int) หรือ string เช่น `"2h 30m 40s"` ต้องน้อยกว่าค่าสูงสุดที่ระบบกำหนด | ไม่ | `None` | `"1h 15m"`                |

### `optimizer_options`
| ชื่อ              | ประเภท     | คำอธิบาย                              | จำเป็น | ค่าเริ่มต้น       |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | จำนวน generation                    | ไม่       | 20            |
| population_size   | int      | ขนาดของ population                    | ไม่       | 20            |
| mutation_range    | list   | ค่า mutation factor สูงสุดและต่ำสุด              | ไม่       | [0, 0.25]     |
| recombination     | float      | ค่า recombination factor                     | ไม่       | 0.4           |
| max_parallel_jobs | int      | จำนวนสูงสุดของ QPU job ที่รันแบบ parallel  | ไม่       | 3             |
| max_batchsize | int      | ขนาด batch สูงสุด  | ไม่       | 200     |

> **Note:** - จำนวน generation ที่ประเมินโดย differential evolution คือ `num_generations` + 1 เนื่องจาก population เริ่มต้นถูกรวมด้วย
> - จำนวน Circuit ทั้งหมดคำนวณเป็น `(num_generations + 1) * population_size`
> - การใช้ population size และจำนวน generation ที่มากขึ้นโดยทั่วไปจะปรับปรุงคุณภาพของผล optimization แต่ไม่แนะนำให้เกิน population size 120 และจำนวน generation มากกว่า 20 (เช่น ``120 * 21 = 2520`` Circuit ทั้งหมด) เพราะจะสร้าง Circuit จำนวนมากเกินไป ซึ่งอาจใช้ทรัพยากรการคำนวณและเวลามาก
> 
> - ฟังก์ชันรองรับการรัน optimization ต่อจากครั้งก่อน และสามารถเพิ่มจำนวน generation ได้เสมอ (โดยใช้ input เดิมยกเว้น `previous_session_id` และเพิ่มค่า ``num_generations``)

> **Note:** ตรวจสอบให้แน่ใจว่าเป็นไปตามข้อจำกัด Qiskit Runtime job
> 
> - Sampler: `sampler_shots <= 10_000_000`
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (สำหรับฟังก์ชันนี้ ทุก term ของ observable commute กัน ดังนั้น `observable_size=1`)
> 
> ดูคำแนะนำ [Job limits](/guides/job-limits) สำหรับข้อมูลเพิ่มเติม

# Output
ฟังก์ชันคืนค่า dictionary สองตัว ได้แก่ dictionary `"result"` ที่มีผลลัพธ์ optimization ที่ดีที่สุด รวมถึง optimal solution และ minimum objective cost ที่เกี่ยวข้อง และ `"metadata"` ที่มีข้อมูลจากผลลัพธ์ทั้งหมดที่ได้ในระหว่างกระบวนการ optimize พร้อมกับ metric ที่เกี่ยวข้อง

Dictionary แรกมุ่งเน้นที่ solution ที่ทำงานได้ดีที่สุด ขณะที่ dictionary ที่สองให้ข้อมูลรายละเอียดเกี่ยวกับ solution ทั้งหมด รวมถึง objective cost และ metric อื่น ๆ ที่เกี่ยวข้อง
## Output dictionaries
| **ชื่อ**    | **ประเภท**                     | **คำอธิบาย**                                                                 | **ตัวอย่าง**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | มีกลยุทธ์การลงทุนตามช่วงเวลา โดยแต่ละ timestamp จะ map ไปยังน้ำหนักการลงทุนเฉพาะสินทรัพย์ (น้ำหนักแต่ละตัวคือจำนวนเงินลงทุนที่ normalize ด้วยจำนวนเงินลงทุนรวม) | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | ข้อมูลที่สร้างระหว่างการวิเคราะห์ รวมถึงผลลัพธ์ ต้นทุน และ metrics    | ดูตัวอย่างด้านล่าง |

### คำอธิบาย dictionary `metadata`
| **ชื่อ**                 | **ประเภท**              | **คำอธิบาย**                                                                                     | **ตัวอย่าง**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | ตัวระบุเฉพาะสำหรับ IBM Quantum Session                          | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | Dictionary ที่มี metrics ต่าง ๆ สำหรับแต่ละ sample ที่ผ่าน postprocess เช่น ต้นทุนหรือ constraints | ดูคำอธิบายด้านล่าง        |
| sampler_counts           | dict[str, int]        | Dictionary ที่ key เป็น bitstring ของผลลัพธ์ที่สุ่มได้และ value เป็นจำนวนครั้งที่พบ | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | List ที่แสดงลำดับการลงทุนของสินทรัพย์ในแต่ละ time step ภายในกลยุทธ์การลงทุน | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | QUBO matrix ของปัญหา                                                                         | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | สรุปเวลาการใช้งาน CPU และ QPU (หน่วยวินาที) ในแต่ละขั้นตอนของกระบวนการ                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |

### คำอธิบาย dictionary `all_samples_metrics`
| **ชื่อ**                | **ประเภท**       | **คำอธิบาย**                                                                                                  | **ตัวอย่าง**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | กลยุทธ์การลงทุนที่ได้จากการ decode quantum states | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | จำนวนครั้งที่แต่ละ investment trajectory ถูกสุ่ม index ตรงกับ `investment_trajectories`                | `[5, 3]`                     |
| objective_costs         | list[float]    | ค่าของ objective function สำหรับแต่ละ investment trajectory เรียงจากต่ำสุดไปสูงสุด                 | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | ประสิทธิภาพที่ปรับตามความเสี่ยง (Sharpe ratio) สำหรับแต่ละ investment trajectory จัดเรียงตาม index                      | `[1.1, 0.7]`                 |
| returns                 | list[float]    | ผลตอบแทนที่คาดหวังสำหรับแต่ละ investment trajectory จัดเรียงตาม index                                               | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | การเบี่ยงเบนสูงสุดของ constraint ภายในแต่ละ investment trajectory จัดเรียงตาม index                               | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | ต้นทุนธุรกรรมโดยประมาณของแต่ละ investment trajectory จัดเรียงตาม index                        | `[0.01, 0.02]`               |

# เริ่มต้นใช้งาน
ยืนยันตัวตนด้วย [API key](https://quantum.cloud.ibm.com/) และเลือก Qiskit Function ดังนี้ (โค้ดนี้สมมติว่าคุณ[บันทึก account](/guides/functions#install-qiskit-functions-catalog-client) ไว้ในเครื่องแล้ว)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## ตัวอย่าง: Dynamic portfolio optimization กับเจ็ดสินทรัพย์
ตัวอย่างนี้แสดงวิธีรัน dynamic portfolio optimization (DPO) function และปรับแต่งการตั้งค่าเพื่อประสิทธิภาพสูงสุด โดยมีขั้นตอนละเอียดสำหรับการ fine-tune พารามิเตอร์เพื่อให้ได้ผลลัพธ์ที่ต้องการ

กรณีนี้ใช้เจ็ดสินทรัพย์ สี่ time steps และสี่ resolution qubits ทำให้ต้องการ Qubit รวมทั้งหมด 112 ตัว

### 1. อ่านสินทรัพย์ที่รวมอยู่ใน portfolio
ถ้าสินทรัพย์ทั้งหมดใน portfolio ถูกเก็บไว้ในโฟลเดอร์ที่ path หนึ่ง สามารถโหลดลง `pandas.DataFrame` และแปลงเป็น object รูปแบบ `dict` ได้ด้วยฟังก์ชันต่อไปนี้

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

ในตัวอย่างนี้ใช้สินทรัพย์ [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) และ [XS2239553048](https://finance.yahoo.com/quote/XS2239553048) รูปต่อไปนี้แสดงข้อมูลที่ใช้ในตัวอย่าง โดยแสดงวิวัฒนาการของราคาปิดรายวันของสินทรัพย์ตั้งแต่วันที่ 1 มกราคม ถึง 1 กันยายน 2023

ในตัวอย่างนี้ เพื่อให้ข้อมูลสอดคล้องกันในทุกวัน เราได้เติมวันที่ไม่ใช่วันซื้อขายด้วยราคาปิดจากวันที่มีข้อมูลก่อนหน้า ทำขั้นตอนนี้เพราะสินทรัพย์ที่เลือกมาจากตลาดต่างกันที่มีวันซื้อขายต่างกัน จึงจำเป็นต้องทำให้ชุดข้อมูลมีมาตรฐานเดียวกัน

![Visualization of the historical data of the assets](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. กำหนดปัญหา

กำหนด specification ของปัญหาโดยตั้งค่าพารามิเตอร์ใน dictionary `qubo_settings`

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. กำหนดการตั้งค่า optimizer และ ansatz (ไม่บังคับ)
ตั้งค่าเฉพาะสำหรับกระบวนการ optimization ได้ตามต้องการ รวมถึงการเลือก optimizer และพารามิเตอร์ของมัน รวมถึงการระบุ primitive และการตั้งค่าต่าง ๆ

สำหรับ Tailored Ansatz ขนาด population size ที่เลือกมาจากการทดลองก่อนหน้าที่แสดงว่าค่านี้ให้ผล optimization ที่เสถียรและมีประสิทธิภาพ

สำหรับ Real Amplitudes Ansatz สามารถใช้ความสัมพันธ์เชิงเส้นระหว่าง ``population_size`` กับจำนวน Qubit ใน Circuit ได้ ตามกฎคร่าว ๆ แนะนำให้ใช้ ``population_size ~ 0.8 * n_qubits`` **ขั้นต่ำ** สำหรับ ansatz `real_amplitudes`

คาดว่า Optimized Real Amplitudes จะมีประสิทธิภาพ optimization ดีกว่า Real Amplitudes ansatz อย่างไรก็ตาม จำนวนตัวแปรที่ต้อง optimize ใน ansatz นี้เพิ่มขึ้นเร็วกว่า Real Amplitudes มาก (ดู[งานวิจัย](https://arxiv.org/pdf/2412.19150)) ดังนั้นสำหรับปัญหาขนาดใหญ่ Optimized Real Amplitudes ต้องการการรัน Circuit มากขึ้น Optimized Real Amplitudes น่าจะเหมาะกับปัญหาที่ต้องการ Qubit ไม่เกิน 100 ตัว แต่ควรระวังในการตั้งค่าพารามิเตอร์ ``population_size`` เพื่อเป็นตัวอย่างของการ scale-up ใน ``population_size`` ตารางก่อนหน้าแสดงว่าสำหรับปัญหา 84 Qubit Optimized Real Amplitudes ต้องการ ``population_size`` 120 ขณะที่ปัญหา 56 Qubit ใช้ ``population_size`` เพียง 40

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

นอกจากนี้ยังสามารถเลือก ansatz เฉพาะได้ ตัวอย่างต่อไปนี้ใช้ ansatz ``'Tailored'``

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. รันปัญหา

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. ดึงผลลัพธ์
ตามที่กล่าวไว้ในส่วน [Output](#output) ฟังก์ชันส่งคืน dictionary ที่มี investment trajectories เรียงจากต่ำสุดไปสูงสุดตามค่า objective function ผลลัพธ์ชุดนี้ช่วยระบุ trajectory ที่มีต้นทุนต่ำสุดและการประเมินการลงทุนที่สอดคล้องกัน นอกจากนี้ยังเปิดโอกาสให้วิเคราะห์ trajectories ทางเลือกต่าง ๆ ทำให้เลือกได้ว่าแบบไหนตรงกับความต้องการหรือเป้าหมายของตัวเองมากที่สุด ความยืดหยุ่นนี้ทำให้ปรับแต่งการเลือกให้เหมาะกับสถานการณ์ที่หลากหลาย

เริ่มต้นด้วยการแสดงกลยุทธ์ผลลัพธ์ที่ได้ objective cost ต่ำสุดที่พบในระหว่างกระบวนการ

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>